In [1]:
from data_processing import load_and_prepare_dataset

df = load_and_prepare_dataset(
    'dataset.csv',
    drop_cols=[
        'fast_break_pct_avg_diff',
        'fast_break_pct_rolling_5_diff',
        'points_off_turnover_pct_avg_diff',
        'points_off_turnover_pct_rolling_5_diff'
    ]
)



In [2]:
import numpy as np

# Correlations with 'team_winner' (exclude 'spread', 'team_winner')
cols_without_spread = [col for col in df.columns if col not in ['spread', 'team_winner']]
corrs_team_winner = []

for col in cols_without_spread:
    if df[col].dtype.kind in 'bifc' and col != 'team_winner':
        try:
            corr = df[col].corr(df['team_winner'])
            if np.isfinite(corr):
                corrs_team_winner.append((col, corr))
        except Exception as e:
            continue

# Sort by absolute value of correlation, descending
corrs_team_winner = sorted(corrs_team_winner, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 most correlated features with 'team_winner':")
for col, corr in corrs_team_winner[:10]:
    print(f"{col}: {corr:.4f}")

print("\n")

# Correlations with 'spread' (exclude 'team_winner', 'spread')
cols_without_team_winner = [col for col in df.columns if col not in ['team_winner', 'spread']]
corrs_spread = []

for col in cols_without_team_winner:
    if df[col].dtype.kind in 'bifc' and col != 'spread':
        try:
            corr = df[col].corr(df['spread'])
            if np.isfinite(corr):
                corrs_spread.append((col, corr))
        except Exception as e:
            continue

corrs_spread = sorted(corrs_spread, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 most correlated features with 'spread':")
for col, corr in corrs_spread[:10]:
    print(f"{col}: {corr:.4f}")


Top 10 most correlated features with 'team_winner':
elo_diff: 0.4574
point_differential_avg_diff: 0.4186
net_eff_avg_diff: 0.3833
margin_estimate: 0.3817
win_loss_pct_diff: 0.3750
point_differential_rolling_5_diff: 0.3700
last_10_efficiency_diff: 0.3668
power_rating_diff: 0.3385
off_eff_avg_diff: 0.3349
ppp_avg_diff: 0.3349


Top 10 most correlated features with 'spread':
elo_diff: 0.5669
point_differential_avg_diff: 0.5449
net_eff_avg_diff: 0.4991
margin_estimate: 0.4989
point_differential_rolling_5_diff: 0.4683
win_loss_pct_diff: 0.4679
last_10_efficiency_diff: 0.4672
off_eff_avg_diff: 0.4321
ppp_avg_diff: 0.4321
power_rating_diff: 0.4313


In [3]:
corr_before = df['margin_estimate'].corr(df['spread'])

best_corr = 0
best_shift = 0

# Try a reasonable range of values for the additive/subtractive shift
for shift in np.arange(0, 10.1, 0.1):
    temp = df['margin_estimate'].copy()
    temp[df['team_home_away'] == 1] += shift
    temp[df['team_home_away'] == 0] -= shift
    corr = temp.corr(df['spread'])
    if abs(corr) > abs(best_corr):
        best_corr = corr
        best_shift = shift

# Apply the best shift found to a new column
df['margin_estimate_adj'] = df['margin_estimate']
df.loc[df['team_home_away'] == 1, 'margin_estimate_adj'] = df['margin_estimate'] + best_shift
df.loc[df['team_home_away'] == 0, 'margin_estimate_adj'] = df['margin_estimate'] - best_shift
corr_after = df['margin_estimate_adj'].corr(df['spread'])
print(f"Best shift: {best_shift:.2f}")

print(corr_before, corr_after)

Best shift: 6.70
0.49886213351821623 0.5625950423228993
